# Full dev evaluation with the revised reader
Attach the `musique_itercomp` repository directory with `data/benchmark/evaluation` and `itercomp_runs_export_evaluation.zip` to this Kaggle notebook. The extracted export folder also works. Enable GPU, Internet, and access to the approved Hugging Face account's `HF_TOKEN` secret. Run the cells in order.

Evaluate Raw documents, Oracle supporting paragraphs, and saved IterCOMP contexts on all **2,417 MuSiQue-Ans dev questions**. All three use the exact reader prompt selected on calibration. The same archived Llama revision, NF4 precision, and decoding settings apply to all three. The IterCOMP compressor is reused from the export. Only the Oracle context builder and scorer receive gold data.

The two archived compression failures stay in the IterCOMP denominator as empty predictions. Raw and Oracle still read those questions. Expect **7,251 prediction rows** and **7,249 reader generations** if there are no additional failures.


In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, hashlib, shutil, subprocess, csv, copy, statistics, tempfile
from zipfile import ZipFile

os.environ.update(USE_TORCH='1', USE_TF='0', USE_FLAX='0')
# Leave paths as None for automatic discovery under /kaggle/input.
SOURCE_REPO = None  # e.g. Path('/kaggle/input/datasets/your-name/your-project/musique_itercomp')
EVALUATION_EXPORT = None  # path to the attached ZIP or extracted folder
DATA_DIR = None  # optional override for the prepared benchmark directory
INPUT_ROOT = Path('/kaggle/input')
REPO = Path('/kaggle/working/musique_itercomp_revised_dev')
OUTPUT_ROOT = Path('/kaggle/working/itercomp_reader_revised_dev')
READER_DEVICE = 'cuda:0'
INSTALL_DEPENDENCIES = True  # use False only when this exact stack is installed already

def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_once(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_text(encoding='utf-8') != text:
            raise ValueError(f'Existing experiment file differs: {path}. Choose a fresh OUTPUT_ROOT.')
    else:
        path.write_text(text, encoding='utf-8')

def save_json(path, value):
    write_once(path, json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + '\n')

def choose_path(explicit, candidates, label):
    if explicit is not None:
        return Path(explicit).resolve()
    candidates = sorted(set(Path(p).resolve() for p in candidates))
    if len(candidates) != 1:
        raise ValueError(f'Set {label} explicitly; found {len(candidates)} candidates: {candidates}')
    return candidates[0]


## Locate the repository and evaluation export
The notebook can find the ZIP by filename or find an extracted run by the dev input checksum. Set `EVALUATION_EXPORT` explicitly if more than one matching copy is attached. The prepared dev files come from `DATA_DIR`, which defaults to the attached repository's `data/benchmark` directory.


In [ ]:
SOURCE_REPO = choose_path(SOURCE_REPO, [
    p.parent for p in INPUT_ROOT.rglob('pyproject.toml')
    if (p.parent/'src/itercomp/artifacts.py').is_file()
    and (p.parent/'scripts/run_experiment.py').is_file()
], 'SOURCE_REPO')
DATA_DIR = Path(DATA_DIR).resolve() if DATA_DIR is not None else SOURCE_REPO/'data/benchmark'
INPUTS, GOLD = (DATA_DIR/'evaluation'/name for name in ('inputs.jsonl', 'gold.jsonl'))
for path in (SOURCE_REPO/'pyproject.toml', INPUTS, GOLD):
    if not path.is_file():
        raise FileNotFoundError(path)

dataset_sha = file_sha(INPUTS)
sources = []
if EVALUATION_EXPORT is None:
    for filename in ('itercomp_runs_export_evaluation.zip', 'itercomp_runs_evaluations.zip'):
        sources.extend(p for p in INPUT_ROOT.rglob(filename) if p.is_file())
    for path in INPUT_ROOT.rglob('run_metadata.json'):
        try:
            meta = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        identity = meta.get('identity', {})
        if (identity.get('stage') == 'compress' and identity.get('method') == 'itercomp'
                and identity.get('dataset_sha256') == dataset_sha
                and meta.get('complete_split') and (path.parent/'compression_results.jsonl').is_file()):
            sources.append(path.parent.parent)
evaluation_source = choose_path(EVALUATION_EXPORT, sources, 'EVALUATION_EXPORT')
if not evaluation_source.exists():
    raise FileNotFoundError(evaluation_source)

if not REPO.exists():
    shutil.copytree(SOURCE_REPO, REPO, ignore=shutil.ignore_patterns(
        '.git', '.venv', 'venv', '__pycache__', 'data', 'results', 'analysis', '*.zip'))
elif not (REPO/'src/itercomp/artifacts.py').is_file():
    raise ValueError(f'{REPO} is not an IterCOMP working copy')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
archive_sha = None
if evaluation_source.is_dir():
    EXTRACTED = evaluation_source
    source_format = 'extracted_directory'
else:
    archive_sha = file_sha(evaluation_source)
    EXTRACTED = OUTPUT_ROOT.parent / ('evaluation_export_' + archive_sha[:16])
    if not EXTRACTED.exists():
        with tempfile.TemporaryDirectory(dir=OUTPUT_ROOT.parent) as temporary:
            unpacked = Path(temporary)/'unpacked'
            with ZipFile(evaluation_source) as archive:
                for member in archive.infolist():
                    name = PurePosixPath(member.filename.replace('\\', '/'))
                    if (not name.parts or name.is_absolute() or '..' in name.parts
                            or any(':' in part for part in name.parts)
                            or (member.external_attr >> 16) & 0o170000 == 0o120000):
                        raise ValueError(f'Unsafe archive member: {member.filename}')
                archive.extractall(unpacked)
            unpacked.rename(EXTRACTED)
    source_format = 'zip'
print('Repository:', SOURCE_REPO)
print('Evaluation export:', evaluation_source, '| format:', source_format)
print('Outputs:', OUTPUT_ROOT)


## Install the reader dependencies
Use a fresh Kaggle session if you have already imported model libraries. The project dependency pins preserve Kaggle's CUDA PyTorch.


In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements-gpu.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO), '--no-deps'], check=True)
sys.path.insert(0, str(REPO/'src'))
sys.path.insert(0, str(REPO))
import importlib
importlib.invalidate_caches()
from itercomp.config import config_from_dict
from itercomp.artifacts import inspect_artifacts, source_hash
from itercomp.dataset import load_inputs
from itercomp.prompt_builder import load_prompts, prompt_hashes
import yaml


## Verify dev artifacts and freeze the reader
The notebook validates the archived compression against this repository and dev data. It checks the paired archived reader predictions and pins the resolved model and tokenizer revisions. The prompt hash must match the revised calibration prompt.


In [ ]:
samples = load_inputs(INPUTS)
sample_ids = [s.id for s in samples]
expected_ids = set(sample_ids)
if len(sample_ids) != 2417 or len(expected_ids) != len(sample_ids):
    raise ValueError('Expected the full 2,417-question evaluation split')
gold_ids = [row['id'] for row in read_jsonl(GOLD)]
if len(gold_ids) != len(sample_ids) or set(gold_ids) != expected_ids:
    raise ValueError('Evaluation inputs and gold IDs differ')

archived_runs = []
for path in sorted(EXTRACTED.rglob('run_metadata.json')):
    meta = json.loads(path.read_text(encoding='utf-8'))
    identity = meta.get('identity', {})
    if (identity.get('dataset_sha256') == dataset_sha and identity.get('method') == 'itercomp'
            and identity.get('selected_ids') == sample_ids and meta.get('complete_split')
            and meta.get('selected_subset_complete') and meta.get('status') == 'complete'):
        archived_runs.append((path.parent, meta))

def unique_run(stage):
    matches = [(folder, meta) for folder, meta in archived_runs if meta['identity']['stage'] == stage]
    if len(matches) != 1:
        raise ValueError(f'Expected one complete dev {stage} run in the evaluation export; found {len(matches)}')
    return matches[0]

compression_dir, compression_meta = unique_run('compress')
archived_reader_dir, archived_reader_meta = unique_run('read')
COMPRESSION_RESULTS = compression_dir/'compression_results.jsonl'
ARCHIVED_PREDICTIONS = archived_reader_dir/'predictions.jsonl'
compression_sha = compression_meta['output_sha256']
archived_reader_identity = archived_reader_meta['identity']
if archived_reader_identity['compression_file_sha256'] != compression_sha:
    raise ValueError('Archived reader and compression runs are from different artifacts')
archived_predictions_sha = file_sha(ARCHIVED_PREDICTIONS)
if archived_predictions_sha != archived_reader_meta['output_sha256']:
    raise ValueError('Archived reader predictions checksum mismatch')

base_data = copy.deepcopy(compression_meta['details']['config'])
expected_source = compression_meta['identity']['compression_contract']['algorithm_source_sha256']
if source_hash() != expected_source:
    raise ValueError('Attach the repository version used for the saved compression.')
original_prompts = load_prompts(config_from_dict(base_data))
original_hashes = prompt_hashes(original_prompts)
if original_hashes['reader'] != archived_reader_identity['reader_prompt_sha256']:
    raise ValueError('Repository reader prompt differs from the archived original prompt')

PROMPT_NAME = 'no_abstention_instruction'
current_prompt = original_prompts['reader']
instruction = ' If the information is insufficient, return exactly INSUFFICIENT_INFORMATION.'
if current_prompt.count(instruction) != 1:
    raise ValueError('Expected exactly one original abstention sentence')
revised_prompt = current_prompt.replace(instruction, '', 1)
revised_hash = hashlib.sha256(revised_prompt.encode('utf-8')).hexdigest()
if revised_hash != '47bf31d8ac64669d711e4bc8719dda06450f629ca4fffddb42c8fd3d1bc7375a':
    raise ValueError('Revised prompt differs from the prompt tested on calibration')

base_data['reader'] = copy.deepcopy(archived_reader_identity['reader'])
reader_init = archived_reader_meta['component_initialization']['reader']
reader_revision = reader_init.get('resolved_revision')
if not reader_revision:
    raise ValueError('Archived reader checkpoint revision is missing')
base_data['reader']['revision'] = reader_revision
base_data['reader']['tokenizer_revision'] = reader_init.get('resolved_tokenizer_revision') or reader_revision
base_data['reader']['device'] = READER_DEVICE
base_data['reference_tokenizer'] = copy.deepcopy(archived_reader_identity['reference_tokenizer'])
if base_data['reader']['backend'] != 'transformers' or base_data['reader']['do_sample']:
    raise ValueError('Expected the archived deterministic Transformers reader')

prompt_path = OUTPUT_ROOT/'prompts'/f'reader_{PROMPT_NAME}.txt'
write_once(prompt_path, revised_prompt)
base_data['prompts']['reader'] = str(prompt_path)
cfg = config_from_dict(base_data)
hashes = prompt_hashes(load_prompts(cfg))
_, available_ids = inspect_artifacts(COMPRESSION_RESULTS, samples, cfg, dataset_sha, hashes)
if available_ids != expected_ids:
    raise ValueError('Saved compression does not cover full dev')

# These previously failed artifacts stay in the full-split run.
from itercomp.dataset import read_jsonl as stream_jsonl
compression_failures = {
    row['id']: row['error'] for row in stream_jsonl(COMPRESSION_RESULTS) if row.get('error')
}
if sorted(compression_failures) != sorted(compression_meta['failed_sample_ids']):
    raise ValueError('Compression failures differ from archived metadata')
CONFIG = OUTPUT_ROOT/'configs'/f'{PROMPT_NAME}.yaml'
write_once(CONFIG, yaml.safe_dump(base_data, sort_keys=False, allow_unicode=True))
METHODS = ('itercomp', 'raw', 'oracle')
RUN_DIRS = {method: OUTPUT_ROOT/'readers'/f'{method}_{PROMPT_NAME}' for method in METHODS}
gold_sha = file_sha(GOLD)
save_json(OUTPUT_ROOT/'experiment_manifest.json', {
    'experiment': 'full_dev_revised_reader', 'n': len(sample_ids), 'sample_ids': sample_ids,
    'conditions': [path.name for path in RUN_DIRS.values()], 'inputs_sha256': dataset_sha,
    'gold_sha256': gold_sha, 'source_sha256': expected_source,
    'compression_sha256': compression_sha, 'compression_failures': compression_failures,
    'archived_predictions_sha256': archived_predictions_sha, 'original_prompt_sha256': original_hashes['reader'],
    'revised_prompt_sha256': revised_hash, 'reader': base_data['reader'],
})
save_json(OUTPUT_ROOT/'provenance/compression_metadata.json', compression_meta)
save_json(OUTPUT_ROOT/'provenance/archived_reader_metadata.json', archived_reader_meta)
save_json(OUTPUT_ROOT/'provenance/evaluation_source.json', {
    'format': source_format, 'archive_sha256': archive_sha,
    'compression_sha256': compression_sha, 'archived_predictions_sha256': archived_predictions_sha,
})
print('Verified dev questions:', len(sample_ids))
print('Inherited IterCOMP compression failures:', sorted(compression_failures))
print('Reader:', cfg.reader.model_id, '| revision:', reader_revision)


## Prepare Raw and Oracle contexts on CPU
Raw contains all supplied paragraphs. Oracle contains just the gold supporting paragraphs. Both use the saved run's source ordering and context formatting. IterCOMP reads directly from the verified compression artifact.


In [ ]:
from contextlib import redirect_stdout
from itercomp.experiment import run_compression

CONTEXT_FILES = {'itercomp': COMPRESSION_RESULTS}
for method in ('raw', 'oracle'):
    directory = OUTPUT_ROOT/'contexts'/method
    with (OUTPUT_ROOT/f'prepare_{method}.log').open('a', encoding='utf-8') as log, redirect_stdout(log):
        path = run_compression(cfg, INPUTS, directory, method=method,
                               gold_path=GOLD if method == 'oracle' else None,
                               resume=(directory/'run_metadata.json').exists())
    meta, ids = inspect_artifacts(path, samples, cfg, dataset_sha, hashes)
    if ids != expected_ids or meta['failed_sample_ids']:
        raise ValueError(f'{method} context preparation is incomplete or has errors')
    CONTEXT_FILES[method] = path
context_sha = {method: file_sha(path) for method, path in CONTEXT_FILES.items()}
print('All three contexts are ready for', len(sample_ids), 'questions.')


## Validate the GPU and reader checkpoint access
Enable the `HF_TOKEN` secret for this notebook in Kaggle. The check and generation processes use the same credential through the environment.


In [ ]:
# Check native bitsandbytes CUDA support in a fresh process.
subprocess.run([sys.executable, '-c',
    "import torch, bitsandbytes as bnb; from bitsandbytes.cextension import lib; "
    "assert torch.cuda.is_available(), 'Enable a Kaggle GPU runtime'; "
    "assert lib is not None and getattr(lib, 'compiled_with_cuda', False), 'bitsandbytes CUDA backend missing'; "
    "print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'bitsandbytes', bnb.__version__)"
], check=True)
from huggingface_hub import HfApi, get_token, get_hf_file_metadata, hf_hub_url
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
token = token.strip() if isinstance(token, str) else get_token()
if not token:
    raise RuntimeError('Enable the HF_TOKEN Kaggle secret for the approved Hugging Face account')
os.environ['HF_TOKEN'] = token
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '0'
HfApi(token=token).whoami()
reader = base_data['reader']
get_hf_file_metadata(hf_hub_url(reader['model_id'], 'config.json', revision=reader['revision']), token=token)
del token
print('Reader authentication and checkpoint configuration access verified.')


## Run the three reader conditions
The saved IterCOMP condition runs first. Each condition loads the pinned reader in a separate process and saves each completed question. Rerun this cell to continue in the same working directory. To resume in a new session, restore the checkpoint ZIP contents into `OUTPUT_ROOT` before running the setup cells, and attach the same repository and evaluation input.

Each completed condition produces a checkpoint ZIP. Per-condition logs are saved alongside predictions. Raw can take longer because its input is much larger.


In [ ]:
from zipfile import ZIP_DEFLATED

def checkpoint_export():
    path = OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + '_export.zip')
    temporary = path.with_suffix('.tmp.zip')
    with ZipFile(temporary, 'w', compression=ZIP_DEFLATED, compresslevel=3) as archive:
        for file in sorted(OUTPUT_ROOT.rglob('*')):
            if file.is_file():
                archive.write(file, file.relative_to(OUTPUT_ROOT).as_posix())
    temporary.replace(path)
    return path

def execute_logged(command, log_path):
    with log_path.open('a', encoding='utf-8') as log:
        proc = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1)
        try:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                if line.startswith('read '):
                    done, total = map(int, line.split()[1].split('/'))
                    if done % 100 == 0 or done == total or line.rstrip().endswith(': error'):
                        print(line.strip(), flush=True)
            status = proc.wait()
        finally:
            if proc.poll() is None:
                proc.terminate()
                proc.wait()
            proc.stdout.close()
    if status:
        raise RuntimeError(f'Reader subprocess exited {status}. See {log_path} and rerun this cell to resume.')

def verified_reader(method):
    directory = RUN_DIRS[method]
    meta = json.loads((directory/'run_metadata.json').read_text(encoding='utf-8'))
    identity = meta['identity']
    if (not meta['complete_split'] or not meta['selected_subset_complete']
            or meta['completed_count'] != len(sample_ids) or identity['selected_ids'] != sample_ids
            or identity['method'] != method or identity['dataset_sha256'] != dataset_sha
            or identity['reader_prompt_sha256'] != revised_hash or identity['reader'] != base_data['reader']
            or identity['compression_file_sha256'] != context_sha[method]
            or meta['component_initialization']['reader']['resolved_revision'] != reader_revision
            or file_sha(directory/'predictions.jsonl') != meta['output_sha256']):
        raise ValueError(f'Incomplete predictions or identity mismatch for {method}')
    return meta

for method in METHODS:
    directory = RUN_DIRS[method]
    metadata_path = directory/'run_metadata.json'
    existing = json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else {}
    if existing.get('status') == 'complete' and existing.get('complete_split'):
        meta = verified_reader(method)
        print('Already complete:', method, '| sample errors:', len(meta['failed_sample_ids']))
        continue
    command = [
        sys.executable, str(REPO/'scripts/run_experiment.py'), '--split', 'evaluation',
        '--stage', 'read', '--config', str(CONFIG), '--inputs', str(INPUTS), '--output-dir', str(directory),
        '--compression-results', str(CONTEXT_FILES[method]),
    ]
    if metadata_path.exists():
        command.append('--resume')
    print('Running:', method, flush=True)
    try:
        execute_logged(command, OUTPUT_ROOT/f'{method}_reader.log')
        meta = verified_reader(method)
        print('Completed:', method, '| sample errors:', len(meta['failed_sample_ids']))
    finally:
        print('Checkpoint export:', checkpoint_export(), flush=True)


## Score full dev and compare with the paper
All metrics use the same 2,417 questions, including failed records. The comparison reports the revised IterCOMP gain over Raw, the gain over the archived original prompt on the same contexts, and the paper's Table 3 values for reference. Compression time is reported separately from reader time. Token ratios use the local reader tokenizer and context text.


In [ ]:
from benchmark import answer_scores, evaluate
from math import isclose

paper = {
    'raw': {'EM': 9.69, 'F1': 19.92, 'ratio': 1.00},
    'oracle': {'EM': 21.53, 'F1': 37.51, 'ratio': 0.15},
    'itercomp': {'EM': 16.67, 'F1': 27.36, 'ratio': 0.14},
}
gold = {row['id']: row for row in stream_jsonl(GOLD)}
table, scored = [], {}
for method in ('raw', 'oracle', 'itercomp'):
    meta = verified_reader(method)
    directory = RUN_DIRS[method]
    predictions = directory/'predictions.jsonl'
    report = directory/'scored'
    provenance = {'predictions_sha256': meta['output_sha256'], 'gold_sha256': gold_sha}
    if not report.exists():
        with tempfile.TemporaryDirectory(dir=directory) as temporary:
            new_report = Path(temporary)/'scored'
            evaluate(GOLD, predictions, new_report)
            save_json(new_report/'score_inputs.json', provenance)
            new_report.rename(report)
    elif json.loads((report/'score_inputs.json').read_text(encoding='utf-8')) != provenance:
        raise ValueError(f'Existing scores differ in input identity: {report}')
    summary = json.loads((report/'summary.json').read_text(encoding='utf-8'))['overall']
    observations = {}
    for row in stream_jsonl(predictions):
        rid = row['id']
        if rid in observations or rid not in gold:
            raise ValueError(f'Unexpected or duplicate prediction: {method}/{rid}')
        inherited = method == 'itercomp' and rid in compression_failures
        if inherited and (row['predicted_answer'] or row['reader_llm_calls'] or row['error'] != compression_failures[rid]):
            raise ValueError(f'Inherited failure differs from compression artifact: {rid}')
        em, f1 = answer_scores(row['predicted_answer'], [gold[rid]['answer'], *gold[rid]['answer_aliases']])
        observations[rid] = (em, f1, bool(row.get('error')),
                             not row['predicted_answer'].strip() and not row.get('error'))
    if set(observations) != expected_ids or summary['n'] != len(expected_ids):
        raise ValueError(f'Score coverage mismatch for {method}')
    em, f1 = (statistics.fmean(value[k] for value in observations.values()) for k in (0, 1))
    if not (isclose(em, summary['answer_em'], abs_tol=1e-12)
            and isclose(f1, summary['answer_f1'], abs_tol=1e-12)):
        raise ValueError(f'Disagreement with evaluator for {method}')
    scored[method] = observations
    inherited_count = len(compression_failures) if method == 'itercomp' else 0
    errors = sum(v[2] for v in observations.values())
    table.append({
        'context': method, 'n': summary['n'], 'EM': 100*em, 'F1': 100*f1,
        'paper_EM': paper[method]['EM'], 'paper_F1': paper[method]['F1'],
        'EM_minus_paper_points': 100*em-paper[method]['EM'],
        'F1_minus_paper_points': 100*f1-paper[method]['F1'],
        'context_ratio_macro': summary['compression_ratio'],
        'context_ratio_token_weighted': summary['weighted_compression_ratio'],
        'paper_context_ratio': paper[method]['ratio'],
        'mean_context_tokens': summary['compressed_context_tokens'],
        'mean_reader_input_tokens': summary['reader_input_tokens'],
        'mean_reader_seconds': summary['reader_seconds'],
        'mean_compression_seconds': summary['compression_seconds'],
        'inherited_compression_errors': inherited_count,
        'new_reader_errors': errors-inherited_count,
        'empty_answers_without_error': sum(v[3] for v in observations.values()),
    })

archived = {}
for row in stream_jsonl(ARCHIVED_PREDICTIONS):
    rid = row['id']
    if rid in archived or rid not in gold:
        raise ValueError(f'Unexpected or duplicate archived prediction: {rid}')
    archived[rid] = answer_scores(row['predicted_answer'], [gold[rid]['answer'], *gold[rid]['answer_aliases']])
if set(archived) != expected_ids:
    raise ValueError('Archived reader coverage differs from revised reader coverage')

paired = []
for label, before, after in (
        ('revised_itercomp_minus_revised_raw', scored['raw'], scored['itercomp']),
        ('revised_oracle_minus_revised_itercomp', scored['itercomp'], scored['oracle']),
        ('revised_itercomp_minus_archived_itercomp', archived, scored['itercomp'])):
    paired.append({
        'contrast': label, 'n': len(sample_ids),
        'EM_delta_points': 100*statistics.fmean(after[r][0]-before[r][0] for r in sample_ids),
        'F1_delta_points': 100*statistics.fmean(after[r][1]-before[r][1] for r in sample_ids),
        'exact_matches_gained': sum(after[r][0] > before[r][0] for r in sample_ids),
        'exact_matches_lost': sum(after[r][0] < before[r][0] for r in sample_ids),
    })

report = {
    'split': 'MuSiQue-Ans dev', 'prompt': PROMPT_NAME, 'conditions': table, 'paired': paired,
    'paper': 'https://aclanthology.org/2026.acl-long.1559.pdf#page=7',
    'paper_itercomp_minus_raw_F1_points': paper['itercomp']['F1']-paper['raw']['F1'],
    'archived_itercomp_EM': 100*statistics.fmean(v[0] for v in archived.values()),
    'archived_itercomp_F1': 100*statistics.fmean(v[1] for v in archived.values()),
    'note': 'Follow-up on the previously inspected full dev split. Paper token counting and exact reader settings may differ.',
}
save_json(OUTPUT_ROOT/'comparison.json', report)
for filename, rows in [('comparison.csv', table), ('paired_changes.csv', paired)]:
    with (OUTPUT_ROOT/filename).open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
print(f"{'Context':<12} {'N':>5} {'EM':>7} {'F1':>7} {'Paper EM':>9} {'Paper F1':>9} {'Errors':>7}")
for row in table:
    errors = row['inherited_compression_errors'] + row['new_reader_errors']
    print(f"{row['context']:<12} {row['n']:>5} {row['EM']:>7.2f} {row['F1']:>7.2f} "
          f"{row['paper_EM']:>9.2f} {row['paper_F1']:>9.2f} {errors:>7}")
print(json.dumps(paired, indent=2))


## Export the completed run
Download the ZIP from Kaggle's output browser. It contains the three prediction files, score reports with per-question and per-hop results, error logs, the frozen config, and `comparison.csv` / `paired_changes.csv`.


In [ ]:
print('Export:', checkpoint_export())
